In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import tarfile
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import mudata as md
import pandas as pd
import scanpy as sc
import scipy as sp
import seaborn as sns
import snapatac2 as snap
import squidpy as sq
from anndata import AnnData
from mudata import MuData
from spatialleiden import spatialleiden, spatialleiden_multiomics

from batch_integration.multisample_utils import (
    preprocess_genes,
    preprocess_neighbors,
    preprocess_scaling,
)
from utils import DATA_DIR, RESULT_DIR, remove_tick_and_label

In [ ]:
spatialmux_dir = DATA_DIR / "SpatialMuxSeq"
spatialmux_results = RESULT_DIR / "SpatialMuxSeq"

seed = 42

In [ ]:
spatialmux_results.mkdir(exist_ok=True, parents=True)

In [ ]:
list(spatialmux_dir.glob("*ME13_50um_3*"))

In [ ]:
SPATIAL_COLUMNS = ["barcode", "in_tissue", "x", "y", "x_pixel", "y_pixel"]


def read_tissue_position(tar: tarfile.TarFile):
    return (
        pd.read_csv(
            tar.extractfile("spatial/tissue_positions_list.csv"),
            index_col=0,
            names=SPATIAL_COLUMNS,
        )
        .astype(int)
        .astype({"in_tissue": bool})
    )

# Load

## RNA

In [ ]:
file = "data/SpatialMuxSeq/GSM8189707_ME13_50um_3_spatial_RNA.tar.gz"
with tarfile.open(file, mode="r:gz") as tar:
    spatial_rna = read_tissue_position(tar)

file = "data/SpatialMuxSeq/GSM8189707_ME13_50um_3_RNA.tar.gz"
with tarfile.open(file, mode="r:gz") as tar:
    with tempfile.TemporaryDirectory() as tmpdir:
        tar.extractall(tmpdir)
        dir = Path(tmpdir) / "E13_50_um_3"
        counts = sp.io.mmread(dir / "matrix.mtx.gz").transpose().tocsr()
        features = pd.read_table(
            dir / "features.tsv.gz",
            names=["gene_id", "gene_name"],
            usecols=[0, 1],
            index_col=0,
        )
        barcodes = pd.read_table(
            dir / "barcodes.tsv.gz", names=["barcode"], index_col=0
        )

barcodes = barcodes.join(spatial_rna)
coord = barcodes[["x", "y"]].to_numpy()
barcodes = barcodes.drop(columns=["x", "y"])

rna = AnnData(X=counts, obs=barcodes, var=features, obsm={"spatial": coord})

del counts, barcodes, features, spatial_rna, coord

rna = rna[rna.obs["in_tissue"]]

In [ ]:
rna.write_h5ad(spatialmux_results / "RNA.h5ad")

## ATAC and CUT&Tag

In [ ]:
GENOME = snap.genome.GRCm38

In [ ]:
def add_spatial_coordinates(adata: AnnData, coord_df: pd.DataFrame):
    adata.obs = adata.obs.join(coord_df)
    adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy()
    adata.obs = adata.obs.drop(columns=["x", "y"])


def load_fragment_file(path: str, coord_df: pd.DataFrame):
    adata = snap.pp.import_fragments(path, chrom_sizes=GENOME, sorted_by_barcode=False)
    add_spatial_coordinates(adata, coord_df)
    adata = adata[adata.obs["in_tissue"]]
    return adata.copy()

In [ ]:
file = "data/SpatialMuxSeq/GSM8189706_ME13_50um_3_spatial.tar.gz"

with tarfile.open(file, mode="r:gz") as tar:
    spatial = read_tissue_position(tar)

In [ ]:
fragment_file = "data/SpatialMuxSeq/GSM8189706_ME13_50um_3_ATAC.fragments.tsv.gz"
atac = load_fragment_file(fragment_file, spatial)
atac.write_h5ad(spatialmux_results / "ATAC.h5ad")

In [ ]:
fragment_file = "data/SpatialMuxSeq/GSM8189706_ME13_50um_3_H3K27me3.fragments.tsv.gz"
h3k27me3 = load_fragment_file(fragment_file, spatial)
h3k27me3.write_h5ad(spatialmux_results / "H3K27me3.h5ad")

In [ ]:
fragment_file = "data/SpatialMuxSeq/GSM8189706_ME13_50um_3_H3K4me3.fragments.tsv.gz"
h3k4me3 = load_fragment_file(fragment_file, spatial)
h3k4me3.write_h5ad(spatialmux_results / "H3K4me3.h5ad")

In [ ]:
del spatial

# Preprocess

## RNA

In [ ]:
# sc.pp.calculate_qc_metrics(rna, inplace=True)

sc.pp.filter_genes(rna, max_counts=200_000)

preprocess_genes(rna, min_cells=50, batch_key=None)
preprocess_scaling(rna, n_pcs=30, seed=seed)

## ATAC and CUT&Tag

In [ ]:
def process_fragments(adata: AnnData):
    # snap.pl.frag_size_distr(adata, interactive=False)

    # snap.metrics.tsse(adata, GENOME)
    # snap.pl.tsse(adata, interactive=False)

    # snap.pp.filter_cells(adata, min_counts=7_000, min_tsse=2.5, max_counts=200_000)

    snap.pp.add_tile_matrix(adata)
    snap.pp.select_features(adata, n_features=250_000)
    snap.tl.spectral(adata, n_comps=30, random_state=seed)

In [ ]:
process_fragments(atac)
process_fragments(h3k27me3)
process_fragments(h3k4me3)

# Multimodal

## Prepare

In [ ]:
modalities = {"RNA": rna, "ATAC": atac, "H3K27me3": h3k27me3, "H3K4me3": h3k4me3}

In [ ]:
for name, mod in modalities.items():
    obsm = "X_pca" if name == "RNA" else "X_spectral"
    preprocess_neighbors(mod, obsm, seed=seed)

In [ ]:
mdata = MuData(modalities)
mdata.var_names_make_unique()

In [ ]:
# spatial coordinates are all equal across modalities
mdata.obsm["spatial"] = mdata.mod["ATAC"].obsm["spatial"].copy()
mdata.obs[["x_pixel", "y_pixel"]] = mdata.mod["ATAC"].obs[["x_pixel", "y_pixel"]].copy()

for mod in mdata.mod.values():
    try:
        del mod.obsm["spatial"]
    except KeyError:
        pass
    try:
        mod.obs = mod.obs.drop(columns=["x_pixel", "y_pixel"])
    except KeyError:
        pass

In [ ]:
mdata.write_h5mu(spatialmux_results / "embryo.h5mu")

## Process

In [ ]:
mdata = md.read_h5mu(spatialmux_results / "embryo.h5mu")

In [ ]:
# spatial neighbors
sq.gr.spatial_neighbors(mdata, coord_type="grid", n_neighs=8)

In [ ]:
resolutions = {
    "H3K27me3": 0.7,
    "H3K4me3": 0.7,
    "ATAC": 0.7,
    "RNA": 1,
}

for mod in mdata.mod:
    sc.tl.leiden(
        mdata.mod[mod],
        resolution=resolutions[mod],
        directed=False,
        flavor="igraph",
        random_state=seed,
    )
    sc.tl.umap(mdata.mod[mod], random_state=seed)
    mdata.mod[mod].obsp["spatial_connectivities"] = mdata.obsp["spatial_connectivities"]
    spatialleiden(
        mdata.mod[mod],
        resolution=(resolutions[mod], resolutions[mod]),
        layer_ratio=0.8,
        directed=(False, False),
        seed=seed,
    )

In [ ]:
layer_weights = {
    "H3K27me3": 0.75,
    "H3K4me3": 0.75,
    "ATAC": 1,
    "RNA": 1.5,
    "spatial": 1,
}

resolutions |= {"spatial": 1}

# non-spatial (hack to avoid installing muon)
spatialleiden_multiomics(
    mdata,
    resolution=resolutions,
    directed=False,
    layer_weights=layer_weights | {"spatial": 0},
    seed=seed,
    key_added="leiden",
)

# spatial
spatialleiden_multiomics(
    mdata,
    resolution=resolutions,
    directed=False,
    layer_weights=layer_weights,
    seed=seed,
)

In [ ]:
import colorcet as cc

scatter_kwargs = dict(size=1, lw=0, legend=False)


def scatter(ax, x, y, hue, aspect=True, **kwargs):
    palette = sns.color_palette(cc.glasbey, n_colors=len(hue.cat.categories))
    sns.scatterplot(
        x=x, y=y, hue=hue, ax=ax, palette=palette, **scatter_kwargs, **kwargs
    )
    remove_tick_and_label(ax)
    if aspect:
        ax.set(aspect=1)

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=5, figsize=(20, 12))
x, y = mdata.obsm["spatial"].T
for i, name in enumerate(mdata.mod):
    ax = axs[0, i]
    umap_x, umap_y = mdata.mod[name].obsm["X_umap"].T

    scatter(axs[0, i], umap_x, umap_y, mdata.mod[name].obs["leiden"], aspect=False)
    scatter(axs[1, i], x, y, mdata.mod[name].obs["leiden"], marker="s")
    scatter(axs[2, i], x, y, mdata.mod[name].obs["spatialleiden"], marker="s")
    axs[0, i].set(title=name)
    if i == 0:
        axs[0, i].set(ylabel="UMAP (Leiden)")
        axs[1, i].set(ylabel="Leiden")
        axs[2, i].set(ylabel="SpatialLeiden")

remove_tick_and_label(axs[0, 4])
_ = axs[0, 4].set(title="all")
scatter(axs[1, 4], x, y, mdata.obs["leiden"], marker="s")
scatter(axs[2, 4], x, y, mdata.obs["spatialleiden"], marker="s")

In [ ]:
fig.savefig(spatialmux_results / "SpatialMuxSeq.png")